# Aldagai kategorikoak
Orain arte bezala, Iowako prezioen datu-multzoarekin ([Kaggleko lehiaketa](https://www.kaggle.com/c/home-data-for-ml-course)) jarraituko dugu lanean, eta horrela ikusiko dugu gure score-a are gehiago hobetu dezakegun. Egoera hauek kudeatzeko hiru estrategia desberdin aztertuko ditugu:
1. Zutabe kategorikoak ezabatzea.
2. Ordinal encoding.
3. One-hot encoding.

Jarraian estrategia bakoitzaren **adibide**-kodea ikus dezakezu:

## 1. estrategia: zutabe kategorikoak ezabatzea
Metodo hau da errazena eta, egia esan, orain arte inplizituki erabili dugun metodoa da. Zutabe horiek esplizituki ezabatu nahi baditugu, `select_dtypes` metodoarekin egin dezakegu, objektu den guztia (zenbakia ez den guztia) ezabatu behar dela adieraziz:

```python
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_valid = X_valid.select_dtypes(exclude=['object'])
```
Zutabe kategorikoak datu-multzotik ezabatu ondoren, sortzen den DataFrame-a bi aldagai berritan gordetzen dugu (*drop_X_train* eta *drop_X_valid*); haiek erabiliko ditugu geroago eredua entrenatzeko.

## 2. estrategia: ordinal encoding

Teknika hau aplikatzeko [OrdinalEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html) klasea erabiliko dugu.

```python
from sklearn.preprocessing import OrdinalEncoder

# Jatorrizko datuen kopia egiten dugu, prozesuan daturik ez galtzeko.
label_X_train = X_train.copy()
label_X_valid = X_valid.copy()

# Zutabe kategorikoen izenekin zerrenda bat sortzen dugu.
object_cols = [col for col in train_X.columns if train_X[col].dtype == "object"]
# Ordinal encoder-a zutabeei aplikatzen diegu.
ordinal_encoder = OrdinalEncoder()
label_X_train[object_cols] = ordinal_encoder.fit_transform(X_train[object_cols])
label_X_valid[object_cols] = ordinal_encoder.transform(X_valid[object_cols])
```

*label_X_train* eta *label_X_valid* aldagaietan train datuak egongo dira, aldagai kategorikoak balio numerikoetara eraldatuta.

## 3. estrategia: one-hot encoding

Kasu honetan, scikit-learn-eko [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) klasea erabiliko dugu, balio egokiak dituzten zutabe berriak sortzeko:

```python
from sklearn.preprocessing import OneHotEncoder

# Encoder-a honako parametro hauekin definitzen dugu:
#      * handle_unknown='ignore' -> balidazio-datuek entrenamendu-datuetan ez dauden klaseak izateagatik
#        errorerik ez gertatzeko.
#      * sparse=False -> sortutako zutabeak NumPy array gisa itzul daitezen.
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
# One-hot encoding aplikatzen diegu zutabe kategorikoei.
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[object_cols]))

# One-hot encoding-ak indizea kendu du; berriro jartzen dugu.
OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

# Zutabe kategoriko GUZTIAK ezabatzen ditugu. Ondoren, one-hot encoding aplikatu diegunak gehituko ditugu.
num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

# Orain one-hot encoding-arekin sortu ditugun zutabeak gehitzen ditugu.
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# Zutabe guztiak string motakoak direla ziurtatzen dugu.
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)
```

## Aurreko ereduaren datuak hasieratzen ditugu

In [1]:
# Beharrezko liburutegiak inportatzen ditugu.
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Datuak kargatzen ditugu.
X = pd.read_csv('data/iowa_data.csv')
X_test = pd.read_csv('data/test.csv')

# Target-a definituta ez duten errenkadak ezabatzen ditugu.
X.dropna(axis=0, subset=['SalePrice'], inplace=True)
# Target-a hautatzen dugu.
y = X.SalePrice
# Prezioa train datuetatik ezabatzen dugu.
X.drop(['SalePrice'], axis=1, inplace=True)

#################
# BALIO FALTAK (MISSING VALUES):
#
# Balio faltei dagokienez, zutabe horiek ezabatzea erabakiko dugu.
# Etorkizunean buruhauste gehiago ekiditeko, azken train eta test datuetatik ezabatzen ditugu biak.
#
cols_with_missing = [col for col in X.columns if X[col].isnull().any()]
X.drop(cols_with_missing, axis=1, inplace=True)
X_test.drop(cols_with_missing, axis=1, inplace=True)

# Train eta test (balidazio) datuetan banatzen ditugu.
train_X, val_X, train_y, val_y = train_test_split(X,y,random_state=1,train_size=0.8, test_size=0.2)

## Estrategien konparazioa

Orain zure txanda da ikasitakoa aplikatzeko. Osatu proposatutako ariketak estrategien konparazioa egiteko:

### 1. ariketa: zutabeak ezabatzea
Ezabatu zutabe kategorikoak eta kalkulatu MAE-a. Zerbait deigarria egiten zaizu balioan? Zergatik?

In [15]:
# Eredua entrenatu eta MAE kalkulatzeko funtzio laguntzailea
def score_dataset(X_tr, X_va, y_tr, y_va):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_va)
    return mean_absolute_error(y_va, preds)

# 1. estrategia: zutabe kategorikoak ezabatu
drop_X_train = train_X.select_dtypes(include='number')
drop_X_valid = val_X.select_dtypes(include='number')

print("Zutabe kopurua aurretik:", train_X.shape[1], "| ondoren:", drop_X_train.shape[1])
mae_drop = score_dataset(drop_X_train, drop_X_valid, train_y, val_y)
print("MAE (zutabe kategorikoak ezabatuta):", round(mae_drop))

Zutabe kopurua aurretik: 61 | ondoren: 34
MAE (zutabe kategorikoak ezabatuta): 16601


**Zure erantzuna:** MAE ≈ **16.601**. Deigarria da 27 zutabe kategoriko ezabatu arren (61 → 34 zutabe) errorea ez dela izugarri handitu: ordinal eta one-hot encoding-arekin lortzen direnak baino ~350–430 gehiago baino ez da (≈ %2-3). Arrazoia: Iowako prezioen informazio gehiena zutabe numerikoetan dago (azalera, kalitate orokorra, eraikuntza-urtea...), eta kategorikoek, balio faltak zituzten zutabeak lehendik kenduta, informazio gutxi gehitzen dute. Hala ere, kategorikoek informazio baliagarria dute (kalitateak, auzoa...), eta hobekuntza txiki bat lortzeko balio dute.

### 2. ariketa: ordinal encoding
Hasi aurretik, arazo bati aurre egin behar diogu: gerta daiteke balidazio-datuetan aldagai kategoriko batzuek entrenamendu-datuetan agertzen ez diren balioak hartzea. Adibidez, inprima dezagun `HeatingQC` zutabea pantailan:

In [17]:
print("HeatingQC zutabean train datuetan agertzen diren balioak:", train_X['HeatingQC'].unique())
print("HeatingQC zutabean balidazio-datuetan agertzen diren balioak:", val_X['HeatingQC'].unique())

HeatingQC zutabean train datuetan agertzen diren balioak: <StringArray>
['Gd', 'TA', 'Ex', 'Fa']
Length: 4, dtype: str
HeatingQC zutabean balidazio-datuetan agertzen diren balioak: <StringArray>
['Ex', 'TA', 'Gd', 'Fa', 'Po']
Length: 5, dtype: str


Arazo hau konpontzeko, balidazio-datuetako balioak train datuetan ere agertzen diren zutabe kategorikoak hautatuko ditugu (train-en balioren bat balidazioan ez agertzeak ez du axola, ez du eredua hautsiko).

In [18]:
# Train datuetan dauden aldagai kategorikodun zutabe guztien zerrenda sortzen dugu.
# pandas 3-n testua "str" motakoa da (ez "object"), beraz ez-numerikoak hautatzen ditugu.
object_cols = [col for col in train_X.columns if not pd.api.types.is_numeric_dtype(train_X[col])]

# Beste zerrenda batean gordetzen ditugu balidazio-datuetako balioak train datuetan ere
# agertzen diren zutabeak. Zutabe hauek EZ dute arazorik emango.
good_label_cols = [col for col in object_cols if 
                   set(val_X[col]).issubset(set(train_X[col]))]
        
# Aldagai kategorikodun zutabe arazotsuen izenak kalkulatzen ditugu, hau da,
# balidazio-datuetan train datuetan agertzen ez diren balioak dituztenak.
bad_label_cols = list(set(object_cols)-set(good_label_cols))

# Lortutakoaren laburpena pantailan erakusten dugu.
print('Ordinal Encoding metodoarekin tratatuko ditugun zutabe kategorikoak:', good_label_cols)
print('')
print('Baztertuko ditugun zutabe kategorikoak:', bad_label_cols)

Ordinal Encoding metodoarekin tratatuko ditugun zutabe kategorikoak: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']

Baztertuko ditugun zutabe kategorikoak: ['HeatingQC', 'Exterior1st', 'RoofMatl']


**Orain zure txanda!** Zutabe arazotsuak identifikatu ondoren, ezabatu eta gainerakoei ordinal encoding aplikatzeko moduan gaude. Datuak aurreprozesatu ondoren, kalkulatu MAE-a haiekin:

In [19]:
from sklearn.preprocessing import OrdinalEncoder

# Zutabe arazotsuak (bad_label_cols) ezabatzen ditugu, eta gainerakoak (good_label_cols) kodetzen ditugu
label_X_train = train_X.drop(bad_label_cols, axis=1)
label_X_valid = val_X.drop(bad_label_cols, axis=1)

ordinal_encoder = OrdinalEncoder()
label_X_train[good_label_cols] = ordinal_encoder.fit_transform(train_X[good_label_cols])
label_X_valid[good_label_cols] = ordinal_encoder.transform(val_X[good_label_cols])

print("Zutabe kopurua:", label_X_train.shape[1])
mae_ordinal = score_dataset(label_X_train, label_X_valid, train_y, val_y)
print("MAE (ordinal encoding):", round(mae_ordinal))

Zutabe kopurua: 58
MAE (ordinal encoding): 16172


**Zure erantzuna:** MAE ≈ **16.172**, 1. estrategiarena baino hobea. 3 zutabe arazotsu baztertu ditugu (`RoofMatl`, `HeatingQC`, `Exterior1st`), balidazioan trainean ez zeuden balioak zituztelako, eta beste 24ri ordinal encoding aplikatu diegu. Kontuz: ordinal encoding-ak ordena bat inposatzen du (alfabetikoa, ez benetakoa), baina zuhaitz-ereduekin (Random Forest) hori ez da arazo handia.

### 3. ariketa: one-hot encoding

Lehenik eta behin, zutabe kategoriko bakoitzak har ditzakeen balio desberdinen kopurua ikusiko dugu:

In [20]:
# Zutabe kategorikoek hartzen dituzten balio desberdinen kopurua kalkulatzen dugu.
object_nunique = list(map(lambda col: train_X[col].nunique(), object_cols))
d = dict(zip(object_cols, object_nunique))

# Goranzko ordenan erakusten ditugu.
sorted(d.items(), key=lambda x: x[1])

[('Street', 2),
 ('Utilities', 2),
 ('CentralAir', 2),
 ('LandSlope', 3),
 ('PavedDrive', 3),
 ('LotShape', 4),
 ('LandContour', 4),
 ('ExterQual', 4),
 ('HeatingQC', 4),
 ('KitchenQual', 4),
 ('MSZoning', 5),
 ('LotConfig', 5),
 ('BldgType', 5),
 ('ExterCond', 5),
 ('RoofStyle', 6),
 ('Foundation', 6),
 ('Heating', 6),
 ('SaleCondition', 6),
 ('RoofMatl', 7),
 ('Functional', 7),
 ('Condition2', 8),
 ('HouseStyle', 8),
 ('Condition1', 9),
 ('SaleType', 9),
 ('Exterior1st', 14),
 ('Exterior2nd', 16),
 ('Neighborhood', 25)]

One-hot encoding-erako zutabeak aldagaiak 10 balio desberdin baino gutxiago hartzen baditu bakarrik sortuko ditugu. Bestela, gure datu-multzoa askoz handiagoa egingo genuke:

In [21]:
# One-hot encoding aplikatuko diegun zutabeak.
low_cardinality_cols = [col for col in object_cols if train_X[col].nunique() < 10]

# Kardinalitate altua dutelako (10 balio desberdin baino gehiago) datu-multzotik ezabatuko ditugun zutabeak.
high_cardinality_cols = list(set(object_cols)-set(low_cardinality_cols))

# Emaitza pantailan erakusten dugu.
print('One-hot encoding aplikatuko zaien zutabe kategorikoak:', low_cardinality_cols)
print('')
print('Kardinalitate altua duten (10 baino handiagoa) eta ezabatuko diren zutabe kategorikoak:', high_cardinality_cols)

One-hot encoding aplikatuko zaien zutabe kategorikoak: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']

Kardinalitate altua duten (10 baino handiagoa) eta ezabatuko diren zutabe kategorikoak: ['Exterior2nd', 'Neighborhood', 'Exterior1st']


**Orain zure txanda!** Kardinalitate altuko aldagai kategorikoak identifikatu ondoren, idatzi horiek tratatzeko eta gainerakoei one-hot encoding aplikatzeko behar den kodea. Ondoren, ohi bezala, kalkulatu lortu duzun MAE-a:

In [22]:
from sklearn.preprocessing import OneHotEncoder

# Kardinalitate altuko zutabeak ezabatu eta kardinalitate baxukoei one-hot encoding aplikatu.
# Oharra: scikit-learn berrietan "sparse=False" parametroa "sparse_output=False" izatera pasa da.
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(train_X[low_cardinality_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(val_X[low_cardinality_cols]))

# One-hot encoding-ak indizea kendu du; berriro jartzen dugu.
OH_cols_train.index = train_X.index
OH_cols_valid.index = val_X.index

# Zutabe kategoriko GUZTIAK ezabatu (kardinalitate altukoak ere, orain berriak gehituko ditugu).
num_X_train = train_X.drop(object_cols, axis=1)
num_X_valid = val_X.drop(object_cols, axis=1)

OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# Zutabe-izen guztiak string motakoak izan behar dira
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

print("Zutabe kopurua:", OH_X_train.shape[1])
mae_onehot = score_dataset(OH_X_train, OH_X_valid, train_y, val_y)
print("MAE (one-hot encoding):", round(mae_onehot))

Zutabe kopurua: 158
MAE (one-hot encoding): 16246


**Zure erantzuna:** MAE ≈ **16.246**. Kardinalitate altuko 3 zutabe ezabatu ditugu (`Exterior2nd`, `Neighborhood`, `Exterior1st`) eta gainerako 24ri one-hot encoding aplikatu diegu: 61 zutabetik 158 zutabera pasa gara.

### Konparazioa
| Estrategia | Zutabe kopurua | MAE |
|---|---|---|
| 1. Zutabe kategorikoak ezabatu | 34 | ≈ 16.601 |
| 2. Ordinal encoding | 58 | ≈ 16.172 |
| 3. One-hot encoding | 158 | ≈ 16.246 |

Hiru emaitzak oso antzekoak dira; ordinal encoding-ak apur bat hobeto funtzionatzen du hemen, eta gainera zutabe gutxiago sortzen ditu. One-hot-ek zutabe asko gehitzen ditu irabazi nabarmenik gabe, ez du ordena faltsurik inposatzen, baina `Neighborhood` bezalako zutabe garrantzitsu bat ezabatu behar izan dugu kardinalitate altuagatik. Diferentzia txikiak direnez, ez dira zertan esanguratsuak izan: split bakarrean eta `random_state` jakin batekin neurtu dira.